### Exhaustive 4: Retrieval, and the Forms Information Travels In

This deep-dive notebook breaks down `4-retrieval.py`, where the model answers a question about your online store by asking **your code** to look in `kb.json`.

**The key point:** The model can only read and write *text*. Your knowledge base is a Python `dict` with keys you can index; the model can never be handed that dict, only a flat run of characters. So every fact in this script is taken apart into text and put back together again as it crosses each boundary — disk to your code, your code to the model, the model back to your code. Retrieval is nothing more than: fetch the right text, and get it into `messages` before the model writes.

The tool loop itself is the same one as `Exhaustive_3-tools.ipynb`, so this notebook does not explain it again. What is new here is **the data**: where it lives, what shape it is in, and what it costs to move.

- **Section 1 — Key, value, dict, JSON**: What is a key/value pair? What is the difference between a Python `dict` and the JSON text that looks just like it? Which words change in the round trip?
- **Section 2 — The script begins, and `search_kb` line by line**: What does `json.load(f)` give back, and why is there also a `json.loads`? And: the function takes a `question` and never uses it — so where does that value actually go?
- **Section 3 — The tool definition and call 1**: What is different here from Exhaustive 3's `get_weather`, and what does `"question": {"type": "string"}` buy you?
- **Section 4 — One fact, five forms**: A single sentence from `kb.json` traced through every form it takes on the way to the model, including the one where JSON ends up nested inside JSON.
- **Section 5 — The id round trip**: `"id": 1` leaves as characters and comes back as an `int` you can look things up with. This is the part of retrieval that is genuinely new.
- **Section 6 — Call 3, and a comment in the course code that isn't true**: `4-retrieval.py` labels the last call "Question that doesn't trigger the tool". It triggered the tool. Also: where `parsed_arguments` comes from.
- **Section 7 — Cheat sheet**: Every format crossing in this one script, sorted by which wall it crosses.
- **Section 8 — Vocabulary, so the words stop colliding**: `object` means two different things, and so do `key`, `field` and `property`.

---

##### The journey of one fact, end to end

Record 1's answer — *"Items can be returned within 30 days of purchase …"* — takes eight forms between the file on your disk and the line of your own code that looks that record back up. The indented rows are the calls that move it from one form to the next; every number is from this notebook's saved runs.

<pre style="font-size: 0.85em; line-height: 1.5; white-space: pre; overflow-wrap: normal; word-break: normal; overflow-x: auto;">
       form                         type              reachable by name?
       ───────────────────────────  ────────────────  ──────────────────────
  ┌ 1  kb.json, characters on disk  <span style="opacity: 0.7">str · 870 chars   </span><span style="opacity: 0.7">no · characters only</span>
  │   open(...) + json.load(f)          <span style="opacity: 0.55">you, in search_kb</span>          <span style="opacity: 0.55">Section 2</span>
  ├ 2  kb, the records as a dict    <span style="opacity: 0.7">dict · 3 records  </span><span style="opacity: 0.7">yes · kb["records"][0]</span>
  │   json.dumps(result)                <span style="opacity: 0.55">you, in the loop</span>           <span style="opacity: 0.55">Section 4</span>
  ├ 3  content, the tool result     <span style="opacity: 0.7">str · 695 chars   </span><span style="opacity: 0.7">no · characters only</span>
  │   set as the message's "content"    <span style="opacity: 0.55">you, in the loop</span>         <span style="opacity: 0.55">Section 4.1</span>
  ├ 4  the tool message you append  <span style="opacity: 0.7">dict · 3 keys     </span><span style="opacity: 0.7">no · holds form 3</span>
  │   json.dumps of the whole body      <span style="opacity: 0.55">the SDK</span>                  <span style="opacity: 0.55">Section 4.1</span>
  ╪<span style="opacity: 0.7">══ the wire · one HTTPS POST · characters only ══════════════════════════</span>
  │   parsed, then tokenised            <span style="opacity: 0.55">OpenAI</span>                   <span style="opacity: 0.55">Section 4.2</span>
  ├ 5  what the model is given      <span style="opacity: 0.7">tokens · 414      </span><span style="opacity: 0.7">no · tokens, not values</span>
  │   writes back, shaped by the schema <span style="opacity: 0.55">the model</span>                <span style="opacity: 0.55">Section 5.1</span>
  ╪<span style="opacity: 0.7">══ the wire, coming back · characters again ═════════════════════════════</span>
  ├ 6  message.content              <span style="opacity: 0.7">str · 178 chars   </span><span style="opacity: 0.7">no · characters only</span>
  │   model_validate_json(content)      <span style="opacity: 0.55">the SDK, in parse()</span>        <span style="opacity: 0.55">Section 5</span>
  ├ 7  message.parsed               <span style="opacity: 0.7">KBResponse        </span><span style="opacity: 0.7">yes · parsed.answer</span>
  │   source: int, so 1 and not "1"     <span style="opacity: 0.55">your own class</span>           <span style="opacity: 0.55">Section 5.1</span>
  └ 8  parsed.source                <span style="opacity: 0.7">int · 1           </span><span style="opacity: 0.7">yes · a key you can use</span>
        <span style="opacity: 0.7">→ record["id"] == parsed.source → kb["records"][0]</span>
</pre>

**Read the last column downwards.** It says *yes* on only three rows — 2, 7 and 8 — and those are exactly the moments the fact is a live Python value sitting inside your process. Everywhere else you are holding characters, and every indented row is a call that turns one into the other.

> **Note on which environment produced the saved outputs.** The outputs stored in the original code cells of this notebook were produced under the **`base`** environment (Python 3.12.3, `openai` 2.1.0, `pydantic` 2.11.7). The other Exhaustive notebooks, and every cell added in this pass, run under **`General_env`** (Python 3.12.12, `openai` 3.14.1, `pydantic` 2.13.5). The visible difference is that `openai` 3.x adds fields the 2.x dumps below don't show: `metadata` and `moderation` at the top level, `text_tokens` inside `completion_tokens_details`, and `cache_write_tokens` / `image_tokens` / `text_tokens` inside `prompt_tokens_details`. Nothing about the story changes; re-running the notebook simply fills those extra keys in. `Supplement_4-retrieval.ipynb` holds a `General_env` run if you want to compare.

> **Note on the cells added in this pass.** Every cell added here is **network-free**. They read `kb.json`, re-use strings that the saved runs already produced, and call SDK functions that make no HTTP request, so you can run the whole of Sections 1, 2, 4, 5 and 6 without spending a single API credit. The original cells (`create`, `parse`) are the only ones that cost anything.

#### 1. Key, value, dict, JSON

##### 1.0 — The whole story (read this before anything else)

- **What this section is:** The four words that every cell in this notebook is built out of, defined once, in order, with nothing assumed.
- **The key point:** A Python `dict` and a piece of JSON text look almost identical on screen, and they are **not the same kind of thing**. The dict is a live structure you can reach into by name. The JSON is a flat run of characters that merely *describes* such a structure. Everything in this script is one of those two, and every `json.` call you will see is a conversion between them.

##### 1.1 — The image to hold on to: assembled furniture and its flat-pack box

One picture carries this whole notebook, so it is worth setting up properly.

- **A Python `dict` is an assembled shelf standing in your room.** You can use it immediately: reach out and put a book on the third shelf. In code, "reach out" is `kb["records"][0]["answer"]`.
- **JSON text is the same shelf as a flat-pack box**: flat panels, plus a printed label on each one saying which panel it is. Nothing is assembled. You cannot put a book on it. But it will go through a doorway, into a van, and through a letterbox — and an assembled shelf will not.
- **`json.dumps(...)` is taking the shelf apart into the flat pack.** `json.dumps` = *dump to string*.
- **`json.loads(...)` is assembling the flat pack back into a shelf.** `json.loads` = *load from string*.
- **The doorways in this script are:** A file on disk, the `content` field of a message, and the network. Only flat packs fit through any of them.

That last line is the reason this notebook exists. `Exhaustive_1-basic.ipynb`, Section 4, made the point for the network — no object crosses it, only bytes. What Section 4 of *this* notebook shows is that the same thing happens **three more times inside your own machine**, before the network is ever involved.

##### 1.2 — A key and a value

- **What a key/value pair is:** Two things stored together, where the first one is the *name* you use to find the second one. `"id": 1` is a pair: `"id"` is the key, `1` is the value.
- **What a `dict` is:** A collection of key/value pairs, written between `{` and `}`. Its name is short for *dictionary*, and that is a fair description — you look a word up by its spelling and get back its meaning.
- **The key point:** A `dict` labels its slots; a `list` numbers them. That is the whole difference. `record["answer"]` asks by name. `records[0]` asks by position. Both appear in `kb.json`, nested inside each other.
- **A word of warning that Section 8 comes back to:** In JSON, a `{...}` block is not called a "dict" — it is called an **object**, which is also the Python word for "any value at all". Two meanings, one word. Keep them apart for now; Section 8 lists every collision like this in one place.

The next cell builds one record by hand, turns it into JSON text, and then asks the text for a key, to show what is lost. The record is a copy of the first entry in `kb.json`, with the long answer shortened.

In [1]:
import json

record = {"id": 1,
          "question": "What is the return policy?",
          "answer": "Items can be returned within 30 days."}

print("1. the object          :", record)
print("2. its type            :", type(record))
print("3. one value, by key   :", record["id"], type(record["id"]))

text = json.dumps(record)

print("4. json.dumps(record)  :", text)
print("5. its type            :", type(text))
print("6. its length          :", len(text), "characters")
print("7. its first character :", repr(text[0]))
print("8. asking the text for a key:")
try:
    text["id"]
except TypeError as e:
    print("   text['id'] ->", type(e).__name__ + ":", e)

back = json.loads(text)
print("9. json.loads(text)    :", back, type(back))
print("10. same as we started?:", back == record)

1. the object          : {'id': 1, 'question': 'What is the return policy?', 'answer': 'Items can be returned within 30 days.'}
2. its type            : <class 'dict'>
3. one value, by key   : 1 <class 'int'>
4. json.dumps(record)  : {"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days."}
5. its type            : <class 'str'>
6. its length          : 102 characters
7. its first character : '{'
8. asking the text for a key:
   text['id'] -> TypeError: string indices must be integers, not 'str'
9. json.loads(text)    : {'id': 1, 'question': 'What is the return policy?', 'answer': 'Items can be returned within 30 days.'} <class 'dict'>
10. same as we started?: True


##### 1.3 — What changes in the round trip

- **The key point:** The structure survives the round trip exactly; the *spelling* does not. Four things get renamed on the way out to JSON, and renamed back on the way in. If you have ever looked at two printouts of the same response and wondered why one says `None` and the other says `null`, this is the entire answer.

What changes:

- **Quotes.** Python is happy with `'id'`; JSON only allows `"id"`. This is why a printed dict is full of single quotes and printed JSON never is.
- **`None` becomes `null`.** Same meaning: nothing here.
- **`True` / `False` become `true` / `false`.** Lowercase, because that is what the JSON standard says.
- **Nothing else.** Strings, integers and floats are written the same way in both, and the nesting is identical.

You have already seen this pair without it being named. In `Exhaustive_3-tools.ipynb`, the two cells that print the same completion — `model_dump_json(indent=2)` and `pprint(model_dump())` — differ in exactly these spots: one shows `"content": null`, the other `'content': None`. It is one object printed in two alphabets.

> **Careful — this is not the same table as the one in Exhaustive 2.** `Exhaustive_2-structured.ipynb`, Section 7, Step 1 has a list that reads `Python str → "string"`, `Python list → "array"`. That one is about **type names inside a schema**: a description of what a field is allowed to hold, with no data in it. The table below is about **values**: the actual data, written two ways. A schema says `"type": "integer"`; a value says `1`.

In [2]:
import json

sample = {"a_string": "hi", "an_int": 1, "a_float": 1.5, "a_bool": True,
          "a_none": None, "a_list": [1, 2], "a_dict": {"k": "v"}}

print("1. the Python dict:")
print("  ", sample)
print()
print("2. the same thing as JSON text:")
print("  ", json.dumps(sample))
print()
print("3. value by value:")
print(f"   {'key':10} {'Python value':14} {'Python type':11} {'JSON value'}")
print(f"   {'-' * 10} {'-' * 14} {'-' * 11} {'-' * 10}")
for key, value in sample.items():
    print(f"   {key:10} {repr(value):14} {type(value).__name__:11} {json.dumps(value)}")

1. the Python dict:
   {'a_string': 'hi', 'an_int': 1, 'a_float': 1.5, 'a_bool': True, 'a_none': None, 'a_list': [1, 2], 'a_dict': {'k': 'v'}}

2. the same thing as JSON text:
   {"a_string": "hi", "an_int": 1, "a_float": 1.5, "a_bool": true, "a_none": null, "a_list": [1, 2], "a_dict": {"k": "v"}}

3. value by value:
   key        Python value   Python type JSON value
   ---------- -------------- ----------- ----------
   a_string   'hi'           str         "hi"
   an_int     1              int         1
   a_float    1.5            float       1.5
   a_bool     True           bool        true
   a_none     None           NoneType    null
   a_list     [1, 2]         list        [1, 2]
   a_dict     {'k': 'v'}     dict        {"k": "v"}


##### 1.4 — Nesting, and the actual shape of `kb.json`

- **The key point:** Values can themselves be dicts or lists, to any depth. `kb.json` is three levels deep, and every index you will write in this notebook is just walking those levels one at a time.

Its shape, with the types named:

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
kb                              <span style="opacity: 0.55">dict, 1 key</span>
└─ "records"                    <span style="opacity: 0.55">list, 3 items</span>
   ├─ [0]                       <span style="opacity: 0.55">dict, 3 keys</span>
   │  ├─ "id"        1                                       <span style="opacity: 0.55">int</span>
   │  ├─ "question"  'What is the return policy?'             <span style="opacity: 0.55">str</span>
   │  └─ "answer"    'Items can be returned within 30 ...'    <span style="opacity: 0.55">str</span>
   ├─ [1]                       <span style="opacity: 0.55">dict, 3 keys</span>  — international shipping
   └─ [2]                       <span style="opacity: 0.55">dict, 3 keys</span>  — payment methods
</pre>

Read `kb["records"][0]["answer"]` left to right against that tree and each step is one line down it: take the `records` value, take item `0`, take its `answer` value.

##### 1.5 — `json.load` versus `json.loads`: the missing `s`

Both appear in this script's neighbourhood, and the one-character difference is easy to lose.

- **`json.loads(text)`** — the `s` is for **string**. It takes characters that are already in memory. This is the one `4-retrieval.py` uses on the model's `arguments`.
- **`json.load(f)`** — no `s`, so: a **file**. It takes an open file object, reads it, and parses it in one step. This is the one inside `search_kb`.
- **`json.dumps(obj)`** → a string. **`json.dump(obj, f)`** → writes straight into a file. Same pattern, other direction. This script uses `dumps` only.
- **The key point:** `load`/`dump` talk to files; `loads`/`dumps` talk to strings. Pass the wrong kind of thing and Python says so immediately, which the next cell shows.

In [3]:
import json

with open("kb.json", "r") as f:
    kb = json.load(f)

print("1. what json.load gave back  :", type(kb))
print("2. its keys                  :", list(kb.keys()))
print("3. type of kb['records']     :", type(kb["records"]))
print("4. how many records          :", len(kb["records"]))
print("5. type of one record        :", type(kb["records"][0]))
print("6. keys of one record        :", list(kb["records"][0].keys()))
print()
print("7. drilling down to one value, one step at a time:")
print("   kb                         ->", type(kb).__name__)
print("   kb['records']              ->", type(kb["records"]).__name__, "of", len(kb["records"]))
print("   kb['records'][0]           ->", type(kb["records"][0]).__name__)
print("   kb['records'][0]['id']     ->", repr(kb["records"][0]["id"]), type(kb["records"][0]["id"]).__name__)
print("   kb['records'][0]['answer'] ->", repr(kb["records"][0]["answer"][:40] + "..."))

1. what json.load gave back  : <class 'dict'>
2. its keys                  : ['records']
3. type of kb['records']     : <class 'list'>
4. how many records          : 3
5. type of one record        : <class 'dict'>
6. keys of one record        : ['id', 'question', 'answer']

7. drilling down to one value, one step at a time:
   kb                         -> dict
   kb['records']              -> list of 3
   kb['records'][0]           -> dict
   kb['records'][0]['id']     -> 1 int
   kb['records'][0]['answer'] -> 'Items can be returned within 30 days of ...'


In [4]:
import json

with open("kb.json", "r") as f:
    print("1. type of f                :", type(f))
    from_file = json.load(f)                      # json.load takes the FILE
print("2. json.load(f)             ->", type(from_file).__name__, "with keys", list(from_file.keys()))

with open("kb.json", "r") as f:
    text = f.read()                               # the file's characters
print("3. f.read()                 ->", type(text).__name__, "of", len(text), "characters")

from_text = json.loads(text)                      # json.loads takes the STRING
print("4. json.loads(text)         ->", type(from_text).__name__, "with keys", list(from_text.keys()))
print("5. same content?            :", from_file == from_text)
print()
print("6. what happens if you swap them:")
with open("kb.json", "r") as f:
    try:
        json.loads(f)
    except TypeError as e:
        print("   json.loads(<file>) ->", type(e).__name__ + ":", e)
try:
    json.load(text)
except AttributeError as e:
    print("   json.load(<str>)   ->", type(e).__name__ + ":", e)

1. type of f                : <class '_io.TextIOWrapper'>
2. json.load(f)             -> dict with keys ['records']
3. f.read()                 -> str of 870 characters
4. json.loads(text)         -> dict with keys ['records']
5. same content?            : True

6. what happens if you swap them:
   json.loads(<file>) -> TypeError: the JSON object must be str, bytes or bytearray, not TextIOWrapper
   json.load(<str>)   -> AttributeError: 'str' object has no attribute 'read'


#### 2. The script begins: imports, client, and `search_kb`

The next two cells are `4-retrieval.py`'s opening lines, unchanged.

- **Changed from the course original:** As the course ships it, `4-retrieval.py` has no `load_dotenv()` — it reads `os.getenv("OPENAI_API_KEY")` directly, so it works only if the key is already in your **system** environment. This repository's copy loads `.env` the way `3-tools.py` does, so both scripts now open identically and `OpenAI()` finds the key by itself.
- **What `requests` is doing gone:** Exhaustive 3 imported it to call the weather API. Here the "external system" is a file on your own disk, so the only library needed to reach it is `json`.

In [ ]:
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [ ]:
load_dotenv()

client = OpenAI()

##### 2.1 — `search_kb`, line by line

```python
def search_kb(question: str):
    with open("kb.json", "r") as f:
        return json.load(f)
```

- **`open("kb.json", "r")`** is a built-in function. It takes a filename and a mode, and returns an open **file object**. `"r"` means *read*, text mode. The file object is not the text; it is a handle positioned at the start of the file, which something else has to read from.
- **`with ... as f:`** binds that handle to the name `f` for the length of the indented block, and — this is the point of `with` — closes the file automatically when the block ends, even if an error is raised inside it. Without `with` you would have to call `f.close()` yourself and remember to do it on the error path too.
- **`json.load(f)`** reads all the characters out of `f` and assembles them into Python values: the flat pack becomes the shelf (1.1). It returns a `dict`.
- **`question: str`** is a **type hint**. It tells a reader, and tools like an editor's autocomplete, that `question` is expected to be a string. Python does not enforce it: `search_kb(42)` runs perfectly happily. And in this function it makes no difference at all, for the reason below.

##### 2.2 — "It takes a `question` and never uses it." Where does that value go?

- **The key point:** Nowhere. `question` is received and then dropped. The function's body never mentions it again, so the same three records come back no matter what the model asks. The docstring says as much — *"This is a mock function for demonstration purposes, we don't search"* — but it is worth seeing rather than being told, because the consequences run through the rest of the notebook.

The next cell calls it three times with three very different questions.

In [ ]:
def search_kb(question: str):
    """
    Load the whole knowledge base from the JSON file.
    (This is a mock function for demonstration purposes, we don't search)
    """
    with open("kb.json", "r") as f:
        return json.load(f)

In [8]:
import json

def search_kb(question: str):
    """
    Load the whole knowledge base from the JSON file.
    (This is a mock function for demonstration purposes, we don't search)
    """
    with open("kb.json", "r") as f:
        return json.load(f)

returns = search_kb("What is the return policy?")
mars    = search_kb("Do you ship to Mars?")
nothing = search_kb("")

print("1. asked about returns  ->", len(returns["records"]), "records, ids", [r["id"] for r in returns["records"]])
print("2. asked about Mars     ->", len(mars["records"]),    "records, ids", [r["id"] for r in mars["records"]])
print("3. asked nothing at all ->", len(nothing["records"]), "records, ids", [r["id"] for r in nothing["records"]])
print("4. all three identical? :", returns == mars == nothing)

1. asked about returns  -> 3 records, ids [1, 2, 3]
2. asked about Mars     -> 3 records, ids [1, 2, 3]
3. asked nothing at all -> 3 records, ids [1, 2, 3]
4. all three identical? : True


##### 2.3 — So the question the model wrote travels a long way to be thrown out

It is worth following the value, because every hop is a format change and this is the smallest example of one in the whole script.

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
the model writes    '{"question":"What is the return policy?"}'      <span style="opacity: 0.55">str</span>
   │  json.loads(tool_call.function.arguments)
   ▼
args                {'question': 'What is the return policy?'}       <span style="opacity: 0.55">dict</span>
   │  search_kb(**args)  — ** turns each key into a keyword argument
   ▼
the parameter       question = 'What is the return policy?'          <span style="opacity: 0.55">str</span>
   │
   ▼
<span style="opacity: 0.7">dropped — the body opens kb.json and ignores it</span>
</pre>

(`**` unpacking is covered in `Exhaustive_1-basic.ipynb`, Section 4, and again in `Exhaustive_3-tools.ipynb`, Section 4, so it is not re-explained here.)

**Why the script still works.** Nothing checks that a tool used its arguments. The model gets *all three* records back and does the filtering itself, while it writes the answer, by reading them and picking the one that matches. The work of searching was not removed; it was moved from your code into the model's context window — which is exactly what makes it expensive, as Section 4 measures.

**And it is why `source` exists.** Because the model saw all three records, "which one did you use?" is a real question with a non-obvious answer. That is what Section 5 is about.

In [9]:
import json

# the exact arguments string the model wrote in this notebook's saved run
arguments = '{"question":"What is the return policy?"}'

print("1. what the model wrote          :", repr(arguments))
print("   its type                      :", type(arguments).__name__)

args = json.loads(arguments)
print("2. after json.loads(arguments)   :", args)
print("   its type                      :", type(args).__name__)
print("   its keys                      :", list(args.keys()))
print()
print("3. search_kb(**args) is the same call as")
print("   search_kb(question='What is the return policy?')")
print()

def search_kb(question: str):
    print("4. inside search_kb, `question`  :", repr(question))
    print("   the next line of the body is  : with open('kb.json', 'r') as f")
    print("   `question` is never mentioned again, so the value stops here.")
    with open("kb.json", "r") as f:
        return json.load(f)

result = search_kb(**args)
print()
print("5. what came back                :", len(result["records"]), "records, ids", [r["id"] for r in result["records"]])

1. what the model wrote          : '{"question":"What is the return policy?"}'
   its type                      : str
2. after json.loads(arguments)   : {'question': 'What is the return policy?'}
   its type                      : dict
   its keys                      : ['question']

3. search_kb(**args) is the same call as
   search_kb(question='What is the return policy?')

4. inside search_kb, `question`  : 'What is the return policy?'
   the next line of the body is  : with open('kb.json', 'r') as f
   `question` is never mentioned again, so the value stops here.

5. what came back                : 3 records, ids [1, 2, 3]


##### 2.4 — Prove it yourself: a `search_kb` that actually searches

- **What this cell is:** A drop-in replacement for `search_kb` that uses the `question` argument. It is deliberately crude — lowercase the words, drop punctuation, drop a handful of very common ones, keep any record sharing a word — because the point is not the search algorithm. The point is what changes downstream once *something* filters.
- **The key point:** The function signature is identical, and the tool definition does not change by one character. The model cannot tell the difference. What changes is how much text ends up in the next request.

Three things to watch in the output:

1. **The same question now returns one record instead of three.**
2. **The tool message shrinks from 695 characters to 234**, and Section 4 turns that into a token count you are billed for.
3. **The Tokyo question returns zero records.** Hold on to that one — Section 6 is about what the model does with a question the knowledge base cannot answer, and a real search is the thing that would have made "there is nothing here" a fact rather than a guess.

> A production retrieval system replaces the keyword match with an embedding search: each record is turned into a vector once, the question is turned into a vector at query time, and you keep the nearest few. The shape of this function stays exactly the same — take a question, return the records worth reading — which is why this crude version is a fair stand-in for the idea.

In [10]:
import json

STOPWORDS = {"what", "is", "the", "do", "you", "a", "to", "of", "in", "for", "are", "our"}

def keywords(sentence):
    """Lowercase the words, drop punctuation, drop the very common ones."""
    return {w.strip("?.,").lower() for w in sentence.split()} - STOPWORDS

def search_kb_real(question: str):
    """A real (if crude) search: keep only records sharing a keyword with the question."""
    with open("kb.json", "r") as f:
        kb = json.load(f)
    wanted = keywords(question)
    hits = [r for r in kb["records"] if wanted & keywords(r["question"])]
    return {"records": hits}

def search_kb_mock(question: str):
    """The course version: ignores `question`, returns everything."""
    with open("kb.json", "r") as f:
        return json.load(f)

question = "What is the return policy?"
print("the question the model wrote:", repr(question))
print("its keywords               :", sorted(keywords(question)))   # sorted: a set prints in any order
print()
for label, fn in [("mock (course code)", search_kb_mock), ("keyword search", search_kb_real)]:
    result = fn(question)
    content = json.dumps(result)
    print(f"{label:19} -> {len(result['records'])} record(s), ids {[r['id'] for r in result['records']]},"
          f" content is {len(content):4} characters")
print()
print("the same search on three different questions:")
for q in ["What is the return policy?", "Do you ship internationally?", "What is the weather in Tokyo?"]:
    hits = search_kb_real(q)["records"]
    print(f"   {q:32} -> ids {[r['id'] for r in hits]}")

the question the model wrote: 'What is the return policy?'
its keywords               : ['policy', 'return']

mock (course code)  -> 3 record(s), ids [1, 2, 3], content is  695 characters
keyword search      -> 1 record(s), ids [1], content is  234 characters

the same search on three different questions:
   What is the return policy?       -> ids [1]
   Do you ship internationally?     -> ids [2]
   What is the weather in Tokyo?    -> ids []


#### 3. The tool definition and call 1

The `tools` list is built exactly like `get_weather`'s in `Exhaustive_3-tools.ipynb`, Section 1, which dissects every key in it and shows the captured request body. Only the parts that are genuinely different here are worth stopping on.

- **One parameter, and it is a `string`.** `get_weather` asked for two `number`s — coordinates the model recalls from training. `search_kb` asks for `"question": {"type": "string"}`, which is not a fact the model looks up but **prose the model composes**. In the saved run the model wrote back `"What is the return policy?"` — the user's own sentence, copied. It could equally have written `"return policy refunds"`. The schema constrains the type, not the wording.
- **`"strict": True` earns its keep twice here.** It constrains generation so `arguments` always parses as `{"question": <some string>}` — and, less obviously, it is also what makes the SDK fill in `parsed_arguments`. Section 6 proves that second half.
- **The `description` is the only prose the model has to judge the tool by**, and this one — *"Get the answer to the user's question from the knowledge base"* — makes no mention of what is *in* the knowledge base. Section 6 is about the consequence.

The `messages` list is two dicts, as in every notebook so far. The `system` message is the only place the words "e-commerce store" appear.

The request body for call 1, captured with a fake HTTP transport so nothing was sent:

<div style="font-size: 0.85em">

```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the return policy?"}
  ],
  "model": "gpt-5-nano",
  "tools": [{
    "type": "function",
    "function": {
      "name": "search_kb",
      "description": "Get the answer to the user's question from the knowledge base.",
      "parameters": {
        "type": "object",
        "properties": {"question": {"type": "string"}},
        "required": ["question"],
        "additionalProperties": false
      },
      "strict": true
    }
  }]
}
```

</div>

**Note what is not in it:** `kb.json`. Not one word of the return policy has been sent. At this point the model knows a tool called `search_kb` exists and nothing whatsoever about your store. `prompt_tokens` for this call was **158**.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_kb",
            "description": "Get the answer to the user's question from the knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the return policy?"},
]

In [13]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user', 'content': 'What is the return policy?'}]

In [ ]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [15]:
completion

ChatCompletion(id='chatcmpl-CNhhApYcS3srzRR1Y6IRZGOejh1xc', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_aKAufIMDeHtzPJpN9MG6ShtI', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')]))], created=1759765544, model='gpt-5-nano-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=29, prompt_tokens=158, total_tokens=187, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [16]:
completion.model_dump()

{'id': 'chatcmpl-CNhhApYcS3srzRR1Y6IRZGOejh1xc',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_aKAufIMDeHtzPJpN9MG6ShtI',
      'function': {'arguments': '{"question":"What is the return policy?"}',
       'name': 'search_kb'},
      'type': 'function'}]}}],
 'created': 1759765544,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 29,
  'prompt_tokens': 158,
  'total_tokens': 187,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}

##### 3.1 — Two printouts of the same object, and why they look different

The two cells above print `completion` two ways, and every difference between them is from Section 1.3:

- **`completion`** on its own is Jupyter echoing the object's `repr`: `ChatCompletion(id='chatcmpl-...', choices=[Choice(...)])`. Class names, Python quoting, `None`.
- **`completion.model_dump()`** is the same data as a `dict`: no class names, `None`, single quotes.
- **`completion.model_dump_json()`**, which this notebook doesn't call here but `Supplement_4-retrieval.ipynb` does, would show that same data as one **string**: `null` and double quotes. Nothing about the response changes; only the alphabet does.

`finish_reason='tool_calls'` and `content=None` mean the model asked for a tool and wrote no answer. `Exhaustive_3-tools.ipynb`, Sections 2 and 3, walks the full response tree and explains why nothing has been retrieved yet; it is the same tree here with `search_kb` in place of `get_weather`.

> **Why this `model_dump()` has fewer keys than Exhaustive 3's.** This output was saved under `openai` 2.1.0 (see the note in the first cell). Under 3.14.1 you also get `metadata` and `moderation` at the top level, plus four extra token-detail keys (`text_tokens` under `completion_tokens_details`; `cache_write_tokens`, `image_tokens` and `text_tokens` under `prompt_tokens_details`). Same response, newer field list.

**The one number worth carrying forward:** `prompt_tokens` is **158**, and `completion_tokens` is **29**. Compare them with call 2's in Section 4.

#### 4. One fact, five forms

- **The key point:** The loop in the next cell is the same tool loop as Exhaustive 3, with `search_kb` swapped in for `get_weather`, so its mechanics — `call_function`, `**args`, the `tool_call_id` pairing — are not repeated here; `Exhaustive_3-tools.ipynb`, Section 4, has them line by line. What *is* new is what `json.dumps(result)` does to your knowledge base: it takes a structure you could index and flattens it into 695 characters that nobody can index, and that is the only form in which the model can ever receive it.

> **Changed from the course original, for the reason Exhaustive 3 was.** The loop below appends the assistant message **once, above the loop**. The course's `4-retrieval.py` appends it *inside*, where it lands in `messages` once per tool call — and the API rejects call 2 with a 400 as soon as the model asks for two tools in one reply. `Exhaustive_3-tools.ipynb`, Section 4, works through that failure with the real error message. With the single `search_kb` call here both placements build exactly the same four-entry history, which is why every output saved below is unaffected by the change; the fix is for the day the model asks for two searches at once.

The next cell follows a single sentence — record 1's answer — through every form it takes between the disk and the wire, and at each step asks the same question: **can you still reach the fact by name?**

In [ ]:
def call_function(name, args):
    if name == "search_kb":
        return search_kb(**args)


messages.append(completion.choices[0].message)  # the model's request: once, not once per tool call

for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [18]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user', 'content': 'What is the return policy?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_aKAufIMDeHtzPJpN9MG6ShtI', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_aKAufIMDeHtzPJpN9MG6ShtI',
  'content': '{"records": [{"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}, {"id": 2, "question": "Do you ship internationally?", "answer": "Yes, we ship to over 50 countries worldwide. International shipping typically takes 7-14 business days 

In [19]:
import json

print("Following ONE fact -- record 1's answer -- through every form it takes.")
print()

with open("kb.json", "r") as f:
    disk_text = f.read()
print("form 1  kb.json, characters on disk         type:", type(disk_text).__name__)
print("        reaching the fact: not possible, it is one run of characters")
print("        first 58 chars   :", repr(disk_text[:58]))
print()

kb = json.loads(disk_text)
print("form 2  the dict json.load hands search_kb  type:", type(kb).__name__)
print("        reaching the fact: kb['records'][0]['answer']")
print("        ->", repr(kb["records"][0]["answer"][:50] + "..."))
print()

content = json.dumps(kb)
print("form 3  json.dumps(result), the tool text   type:", type(content).__name__)
print("        reaching the fact: not possible, characters again")
print("        length           :", len(content), "characters")
print()

tool_message = {"role": "tool", "tool_call_id": "call_aKAu...", "content": content}
print("form 4  the tool message you append         type:", type(tool_message).__name__)
print("        reaching the fact: tool_message['content'] -- gets you the characters, not the fact")
print("        keys             :", list(tool_message.keys()))
print()

wire = json.dumps({"messages": [tool_message]})
print("form 5  the request body the SDK sends      type:", type(wire).__name__)
print("        reaching the fact: not possible; note the \\\" escapes below")
print("        a slice of it    :", repr(wire[60:148]))

Following ONE fact -- record 1's answer -- through every form it takes.

form 1  kb.json, characters on disk         type: str
        reaching the fact: not possible, it is one run of characters
        first 58 chars   : '{\n    "records": [\n        {\n            "id": 1,\n        '

form 2  the dict json.load hands search_kb  type: dict
        reaching the fact: kb['records'][0]['answer']
        -> 'Items can be returned within 30 days of purchase w...'

form 3  json.dumps(result), the tool text   type: str
        reaching the fact: not possible, characters again
        length           : 695 characters

form 4  the tool message you append         type: dict
        reaching the fact: tool_message['content'] -- gets you the characters, not the fact
        keys             : ['role', 'tool_call_id', 'content']

form 5  the request body the SDK sends      type: str
        reaching the fact: not possible; note the \" escapes below
        a slice of it    : '", "content": "{\\"

##### 4.1 — Why `content` has to be a string, and what that does to the quotes

- **What it is:** Every message in `messages` has a `content` field, and the API defines that field as **text**. A tool result is a message like any other, so a `dict` cannot go in it. `json.dumps(result)` is what makes it legal.
- **The key point:** That leaves JSON text sitting inside a field of a structure that is itself about to become JSON text. The inner quotes then have to be marked as "these are characters, not the end of my string" — which is what a backslash does. That is the whole explanation for the `\"` storm in any captured request body.

The three layers, each one a `json.dumps` of the layer above:

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
layer 1   the records, as a dict          {'records': [{'id': 1, ...}]}
             │ json.dumps  <span style="opacity: 0.7">← you call this, in the loop</span>
             ▼
layer 2   the tool content, as text       '{"records": [{"id": 1, ...}]}'
             │ placed as the value of "content" in the message dict
             ▼
layer 3   the message, as a dict          {'role': 'tool', 'content': '{"records": ...'}
             │ json.dumps  <span style="opacity: 0.7">← the SDK calls this, on the whole request body</span>
             ▼
          what goes on the wire           {"role": "tool", "content": "{\"records\": [{\"id\": 1, ..."}
                                                                       <span style="opacity: 0.7">▲ layer 2's quotes, escaped</span>
</pre>

- **The backslashes are not in the data.** They are added by layer 3's `json.dumps` and removed by the matching parse on OpenAI's side. `len(content)` is 695 characters at layer 2; the escaped version inside layer 3 is longer, but the model is shown the 695.
- **This is why reading a captured request body is unpleasant** and why Exhaustive 3 flagged the same `\"` in its Section 5 dump. It is presentation, not content.

##### 4.2 — What the whole knowledge base costs

The two calls' `prompt_tokens`, straight off the saved outputs:

| | call 1 (`create`) | call 2 (`parse`) |
| --- | --- | --- |
| what the request carries | system, user, `tools` | the same, plus the assistant tool-call message, plus the tool result, plus the `KBResponse` schema |
| `prompt_tokens` | 158 | 414 |

**+256 tokens**, and the bulk of that is `kb.json` arriving as text. The same two numbers appear in `Supplement_4-retrieval.ipynb`, which was run separately under `openai` 3.14.1 — the jump is reproducible, not a fluke of one run.

Three things follow, and they are the reason real retrieval systems are built the way they are:

- **You pay for the whole knowledge base on every single question**, because `search_kb` returns all of it every time. Three records is nothing. Ten thousand records will not fit in the context window at all, at any price.
- **The keyword search in Section 2.4 cut the tool content from 695 characters to 234** for the same question, by sending one record instead of three. That saving is the entire economic argument for retrieval: not "find the answer", but *"send only what is worth reading"*.
- **A second call always costs more than the first**, even with a tiny tool result, because the request now also carries the model's own tool-call message and the response schema. The API remembers nothing between calls, so each request restates everything (`Exhaustive_3-tools.ipynb`, Section 6).

#### 5. The id round trip: a key that leaves as text and comes back as a number

- **What this section is:** The one mechanism in `4-retrieval.py` that has no counterpart in Exhaustive 1–3. `KBResponse` asks for two fields, and the second one, `source`, is not information the model produces — it is a **key from your own data, handed out and then handed back**.
- **The key point:** `"id": 1` starts life as an `int` in `kb.json`, gets flattened into characters along with everything else, is read by the model as tokens, is written back out by the model as the characters `1`, and is turned back into an `int` — because *your class says* `source: int`. At that moment it stops being a number and becomes a **key you can look up with**, which is how you get from the model's answer back to the record it came from.

The full circle:

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
kb.json          "id": 1                              <span style="opacity: 0.55">int, in a dict on your disk</span>
   │  json.dumps, inside the loop
   ▼
tool content     '... {"id": 1, "question": ...'      <span style="opacity: 0.55">str — now just two characters, a colon and a 1</span>
   │  the request; the model reads it
   ▼
the model        <span style="opacity: 0.7">picks the record that answers the question and notes its id</span>
   │  constrained by "source": {"type": "integer"} in the schema
   ▼
message.content  '{"answer":"Items can be ...","source":1}'   <span style="opacity: 0.55">str — characters again</span>
   │  parse() → KBResponse.model_validate_json(content)
   ▼
message.parsed   KBResponse(answer='Items can be ...', source=1)
   │  source: int in your class is what makes it an int and not the text "1"
   ▼
your code        kb["records"][0]  ── found by matching record["id"] == parsed.source
</pre>

##### 5.1 — Why `source: int` rather than `source: str`

- **What the annotation does:** It is the same mechanism as `temperature: float` in Exhaustive 3. Pydantic writes `"type": "integer"` into the schema (the captured `response_format` for this call is below), which constrains the model to write digits with no quotes; then, coming back, Pydantic builds `source` as a real `int`.
- **The key point:** `record["id"] == parsed.source` is comparing an `int` with an `int`. Had `source` been declared `str`, the model would have written `"1"` and that comparison would be `1 == "1"`, which is `False` — silently, with no error. The annotation is not documentation; it is what makes the lookup work.

The `response_format` the SDK built from `KBResponse` and sent, captured offline:

<div style="font-size: 0.85em">

```json
"response_format": {
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": {
        "answer": {"description": "The answer to the user's question.", "title": "Answer", "type": "string"},
        "source": {"description": "The record id of the answer.",      "title": "Source", "type": "integer"}
      },
      "required": ["answer", "source"],
      "title": "KBResponse",
      "type": "object",
      "additionalProperties": false
    },
    "name": "KBResponse",
    "strict": true
  }
}
```

</div>

`"type": "integer"` came from `source: int`, and the `description` came from `Field(description=...)`. `Exhaustive_3-tools.ipynb`, Section 6, traces that conversion function by function; it is identical here.

**The `Field` description is doing more work than it looks.** *"The record id of the answer"* is the only thing telling the model that `source` refers to the `id` key in the records it was shown. Nothing else connects the two — not the field name, not the type. Take that sentence away and `source` is an integer with no stated meaning.

In [ ]:
class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")


completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=KBResponse,
)

In [21]:
completion_2.model_dump()

{'id': 'chatcmpl-CNhiYKkKdNJZGmaRJLr7m1Mxucfhk',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '{"answer":"Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.","source":1}',
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': None,
    'parsed': {'answer': 'Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.',
     'source': 1}}}],
 'created': 1759765630,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 436,
  'prompt_tokens': 414,
  'total_tokens': 850,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'rea

In [22]:
final_response = completion_2.choices[0].message.parsed
print(final_response.answer)
print(final_response.source)

Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.
1


In [23]:
import json

# the two halves of the round trip, both taken from this notebook's saved outputs
content = completion_2.choices[0].message.content        # what the model wrote, as characters
parsed  = completion_2.choices[0].message.parsed         # the same thing, as an object

print("1. message.content  :", type(content).__name__, "-- characters and nothing else")
print("   its last 11 chars:", repr(content[-11:]))
print()
print("2. message.parsed   :", type(parsed).__name__)
print("   parsed.source    :", repr(parsed.source), "->", type(parsed.source).__name__)
print("   parsed.answer    :", repr(parsed.answer[:45] + "..."))
print()

with open("kb.json", "r") as f:
    kb = json.load(f)

print("3. using parsed.source as a key back into kb.json:")
for record in kb["records"]:
    if record["id"] == parsed.source:
        print("   record", record["id"], "question:", record["question"])
        print("   record", record["id"], "answer  :", record["answer"][:45] + "...")
print()
print("4. is the model's answer a verbatim copy of that record?")
cited = next(r for r in kb["records"] if r["id"] == parsed.source)
print("   parsed.answer == cited['answer'] :", parsed.answer == cited["answer"])

1. message.content  : str -- characters and nothing else
   its last 11 chars: '"source":1}'

2. message.parsed   : KBResponse
   parsed.source    : 1 -> int
   parsed.answer    : 'Items can be returned within 30 days of purch...'

3. using parsed.source as a key back into kb.json:
   record 1 question: What is the return policy?
   record 1 answer  : Items can be returned within 30 days of purch...

4. is the model's answer a verbatim copy of that record?
   parsed.answer == cited['answer'] : True


##### 5.2 — "The model already gave me the answer. Why do I want an id as well?"

- **The key point:** Because `answer` is text the model *wrote*, and `source` is a claim you can *check*. One is output, the other is evidence, and only the second one is mechanically verifiable against data you control.

What the id buys you, in order of how much it matters:

- **You can show the user where it came from.** This is the "Sources" list under an answer in every retrieval product you have used. It needs a key, and `source` is that key.
- **You can catch a wrong record.** If the model answers a shipping question while citing record 1, the mismatch is visible without anyone reading the answer.
- **You can catch a paraphrase, and there will be one.** This is the interesting part, and the next cell demonstrates it with the two runs that actually exist in this repository.
- **You can act on it.** Log which records get used, find the ones that never do, notice when one record answers everything.

##### 5.3 — The two saved runs disagree, and that is the lesson

Both runs asked the same question, got the same three records and cited `source: 1`. Their `answer` strings are not the same:

- **This notebook's saved run** copied record 1 **word for word**.
- **`Supplement_4-retrieval.ipynb`'s run** inserted one word — *"with **the** original receipt"*.

Neither is wrong. But it settles a question worth settling: `answer` is **not** guaranteed to be a substring of your knowledge base. The model is writing prose, and it is free to re-word. If your product needs the exact policy text, take it from `kb.json` using `source` and show *that*; use `answer` for the conversational wrapper.

##### 5.4 — The limit: `strict` guarantees the shape, not the truth

- **The key point:** `"type": "integer"` forces the model to write an integer. It cannot force it to write an integer that **exists**. JSON Schema has no way to say "one of the ids currently in this file", and nothing on OpenAI's side has read your file.

So `{"answer": "...", "source": 99}` is a perfectly valid `KBResponse`, and `parse()` will build it for you without complaint — as the last part of the next cell shows. Checking that the id is real is your job:

```python
by_id = {r["id"]: r for r in kb["records"]}
if parsed.source not in by_id:
    ...                      # the citation is fabricated; don't display it as a source
```

**The general shape of this, worth keeping:** Structured outputs guarantee you a well-formed value of the right type. They guarantee nothing about whether that value is *correct*. Every check that depends on your data is code you have to write.

In [24]:
import json
from pydantic import BaseModel, Field

class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")

with open("kb.json", "r") as f:
    kb = json.load(f)
by_id = {r["id"]: r for r in kb["records"]}          # a dict keyed by id, for lookups

# the two strings the model actually produced, in this notebook and in Supplement_4
runs = {
    "Exhaustive_4's saved run": '{"answer":"Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.","source":1}',
    "Supplement_4's saved run": '{"answer":"Items can be returned within 30 days of purchase with the original receipt. Refunds will be processed to the original payment method within 5-7 business days.","source":1}',
}

for label, content in runs.items():
    answer = KBResponse.model_validate_json(content)
    cited = by_id[answer.source]["answer"]
    print(f"{label}:")
    print("   source            :", answer.source)
    print("   verbatim copy?    :", answer.answer == cited)
    if answer.answer != cited:
        import difflib
        changes = [w for w in difflib.ndiff(cited.split(), answer.answer.split()) if w[0] in "+-"]
        print("   word-level diff   :", changes)
    print()

print("Can the model cite an id that doesn't exist?")
invented = KBResponse.model_validate_json('{"answer":"We refund in gold bullion.","source":99}')
print("   it parses cleanly  :", invented)
print("   is 99 a real id?   :", invented.source in by_id)
print("   the real ids are   :", sorted(by_id))

Exhaustive_4's saved run:
   source            : 1
   verbatim copy?    : True

Supplement_4's saved run:
   source            : 1
   verbatim copy?    : False
   word-level diff   : ['+ the']

Can the model cite an id that doesn't exist?
   it parses cleanly  : answer='We refund in gold bullion.' source=99
   is 99 a real id?   : False
   the real ids are   : [1, 2, 3]


#### 6. Call 3, and a comment in the course code that isn't true

`4-retrieval.py` introduces its last block with:

```python
# Question that doesn't trigger the tool
```

- **The key point:** It triggered the tool. The saved output below has `finish_reason='tool_calls'`, `content=None`, and a request to run `search_kb` with `{"question":"What is the weather in Tokyo?"}`. The comment describes an outcome the author hoped for, not a rule the API enforces. Whether to call a tool is the model's choice, and it is a probabilistic one — `Supplement_4-retrieval.ipynb` records two runs on 2026-09-19 that came out differently from each other.

##### 6.1 — Why it fired

Read what the model was actually given, and the choice stops looking strange:

- **The tool's description is unconditional.** *"Get the answer to the user's question from the knowledge base."* There is no "only for questions about the store", and no list of what the knowledge base contains. Taken at face value, the tool claims it can answer the user's question — whatever the question is.
- **The system message names the domain, but does not forbid anything.** *"...answers questions from the knowledge base about our e-commerce store."* A model weighing "look first, then answer" against "refuse without looking" can reasonably pick the first.
- **Nothing tells it what is inside.** The model has never seen `kb.json`. It cannot know the knowledge base has three records and none of them mention weather. Calling the tool is how it would find out.
- **It thought hard about it:** `reasoning_tokens` is 448 out of 478 completion tokens, against 0 in this notebook's call 1. The choice was not made casually. Reasoning effort varies from run to run — `Supplement_4-retrieval.ipynb`'s call 1 spent 64 — so read the size of the gap rather than the exact figures.

##### 6.2 — What happens next, which is nothing

The script stops here. There is no loop, so the tool request is never run, `content` is `None` — the last cell prints exactly that — and no answer is ever produced. That is not an error; it is the script ending mid-round-trip.

A real agent would do what Exhaustive 3's Section 6 describes: check `finish_reason`, and while it is `'tool_calls'`, run the tools, append the results, and call again. Here, one more turn would have run `search_kb`, handed the model three records about returns, shipping and payment, and let it say the knowledge base has nothing on Tokyo weather.

##### 6.3 — How you would actually stop it firing

Ranked by how much they help:

1. **Say what is in the knowledge base, in the tool description.** *"Search the store's FAQ, which covers returns, international shipping and payment methods. Use only for questions about this store."* The model's one piece of prose about the tool is the cheapest place to fix this.
2. **Give the tool a way to say "nothing found".** The keyword search in Section 2.4 returns `{"records": []}` for the Tokyo question. An empty result is a fact the model can report; with the mock, three irrelevant records are all it ever gets.
3. **Take the choice away with `tool_choice`.** `tool_choice="none"` forbids tools for that call, `tool_choice="required"` demands one, and naming a specific function forces that one. A blunt instrument, but it is a guarantee rather than a hint.
4. **Tighten the system message** — *"If the knowledge base does not cover the question, say so; do not answer from your own knowledge."* Helpful, and the weakest of the four, because it is a request rather than a constraint.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the weather in Tokyo?"},
]

completion_3 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [26]:
completion_3.model_dump()

{'id': 'chatcmpl-CNhjYwgzXmlubkn9HtVJc3tKJuPMG',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_THkMSODNUkJe0XWTkTcNEwQx',
      'function': {'arguments': '{"question":"What is the weather in Tokyo?"}',
       'name': 'search_kb',
       'parsed_arguments': {'question': 'What is the weather in Tokyo?'}},
      'type': 'function'}],
    'parsed': None}}],
 'created': 1759765692,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 478,
  'prompt_tokens': 159,
  'total_tokens': 637,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 448,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, '

In [27]:
print(completion_3.choices[0].message.content)

None


##### 6.4 — `parsed_arguments`, and why it appears only in this call

Look at `completion_3`'s tool call in the dump above and there is a key that call 1's did not have:

```python
"function": {
  "arguments": "{\"question\":\"What is the weather in Tokyo?\"}",
  "name": "search_kb",
  "parsed_arguments": {"question": "What is the weather in Tokyo?"}
}
```

- **What it is:** The same arguments, already run through `json.loads` for you. `arguments` is the string the model wrote; `parsed_arguments` is that string as a `dict`.
- **Why call 1 doesn't have it:** Call 1 used `create()`. Call 3 used `parse()`. `parsed_arguments` is added by the SDK's parsing layer, and `create()` doesn't run it. It is the same split as `message.parsed`, which `parse()` adds and `create()` does not — one helper for the answer, one for the arguments.
- **Where it comes from:** `parse()` hands the finished `ChatCompletion` to `_parse_chat_completion`, which rebuilds each tool call with one extra key. The real SDK source, trimmed to the lines that matter:

```python
# openai/lib/_parsing/_completions.py  —  inside _parse_chat_completion
for tool_call in message.tool_calls:
    if tool_call.type == "function":
        tool_call_dict = tool_call.to_dict()
        tool_calls.append(
            construct_type_unchecked(
                value={
                    **tool_call_dict,
                    "function": {
                        **cast(Any, tool_call_dict["function"]),
                        "parsed_arguments": parse_function_tool_arguments(
                            input_tools=input_tools, function=tool_call.function
                        ),
                    },
                },
                type_=ParsedFunctionToolCall,
            )
        )

# openai/lib/_parsing/_completions.py  —  the function it calls
def parse_function_tool_arguments(*, input_tools, function):
    input_tool = get_input_tool_by_name(input_tools=input_tools, name=function.name)
    if not input_tool:
        return None                       # the tool wasn't in the list you passed
    input_fn = cast(object, input_tool.get("function"))
    if isinstance(input_fn, PydanticFunctionTool):
        return model_parse_json(input_fn.model, function.arguments)
    input_fn = cast(FunctionDefinition, input_fn)
    if not input_fn.get("strict"):
        return None                       # ◄── no "strict" in YOUR tool dict: nothing is parsed
    return json.loads(function.arguments)  # ◄── this is the whole of it
```

- **The key point, and it is easy to miss:** `parsed_arguments` is filled in **only because your tool dict has `"strict": True`**. Drop that key and the field stays `None` — the SDK will not parse arguments it has no guarantee about the shape of. This is the second job `"strict": True` does, and it happens entirely on your machine, after the reply has arrived.
- **Also note `ParsedFunctionToolCall`** in the type of `completion_3`'s tool call, where call 1 had `ChatCompletionMessageFunctionToolCall`. It is a subclass that adds this one field: `class ParsedFunction(Function): parsed_arguments: Optional[object] = None`, in `openai/types/chat/parsed_function_tool_call.py`.

The next cell calls `parse_function_tool_arguments` directly with three different tool lists. It imports from a private module (the leading underscores), which is fine for looking at and not something to build on. No network request is made.

In [28]:
from openai.lib._parsing._completions import parse_function_tool_arguments
from openai.types.chat.chat_completion_message_function_tool_call import Function

# the function object exactly as call 3 came back with it
function = Function(name="search_kb", arguments='{"question":"What is the weather in Tokyo?"}')

strict_tool = [{"type": "function", "function": {
    "name": "search_kb", "description": "Get the answer to the user's question from the knowledge base.",
    "parameters": {"type": "object", "properties": {"question": {"type": "string"}},
                   "required": ["question"], "additionalProperties": False},
    "strict": True}}]

loose_tool = [{"type": "function", "function": {
    "name": "search_kb", "description": "Get the answer to the user's question from the knowledge base.",
    "parameters": {"type": "object", "properties": {"question": {"type": "string"}},
                   "required": ["question"], "additionalProperties": False}}}]   # no "strict" key

print("1. function.arguments, as it arrives :")
print("  ", repr(function.arguments))
print()
print("2. parsed_arguments, for three tool lists:")
print("   with \"strict\": True  ->", repr(parse_function_tool_arguments(input_tools=strict_tool, function=function)))
print("   with no \"strict\" key ->", repr(parse_function_tool_arguments(input_tools=loose_tool,  function=function)))
print("   with no tools at all ->", repr(parse_function_tool_arguments(input_tools=[],          function=function)))

1. function.arguments, as it arrives :
   '{"question":"What is the weather in Tokyo?"}'

2. parsed_arguments, for three tool lists:
   with "strict": True  -> {'question': 'What is the weather in Tokyo?'}
   with no "strict" key -> None
   with no tools at all -> None


#### 7. Cheat sheet: every format crossing in this one script

`Exhaustive_2-structured.ipynb`, Section 7, sorts the JSON-ish calls by **what goes in and what comes out** — schema versus values. This sheet sorts the same kind of calls a different way: by **which wall each one crosses**. A wall is any boundary that only characters can pass through. Knowing which wall you are at tells you which direction you need and therefore which call.

There are three walls in `4-retrieval.py`, and one set of calls that crosses no wall at all.

##### Wall 1 — the disk

Characters in a file on one side, live Python values on the other.

- **`open("kb.json", "r")`** — Built-in. Filename in, an open **file object** out. Not the text yet; a handle to read it from. Used inside `search_kb`.
- **`json.load(f)`** — `json`, inbound. A **file object** in, a `dict` (or list) out. Reads and parses in one step. Used inside `search_kb`. **The key point:** No `s`, so it takes a file.
- **`json.dump(obj, f)`** — `json`, outbound. Not used in this script; it is what you would call to write `kb.json` back out.

##### Wall 2 — the model's reply arriving

Everything the model produces arrives as characters. Three calls turn those characters into things you can use, and two of them the SDK has already run for you.

- **`json.loads(tool_call.function.arguments)`** — `json`, inbound. A **string** in, a `dict` out. You call this yourself, in the loop. **The key point:** The `s` is for string.
- **`tool_call.function.parsed_arguments`** — Not a call; an attribute. The same `json.loads`, already run by the SDK. Present only when you used `parse()` **and** your tool dict has `"strict": True` (Section 6.4). `None` otherwise.
- **`message.parsed`** — Not a call; an attribute. `message.content` already run through your Pydantic class. Present only when you used `parse()` with a `response_format`. It is `KBResponse.model_validate_json(content)` done for you. **The key point:** `content` and `parsed` are the same data in two forms — text and object.

##### Wall 3 — the request leaving

- **`json.dumps(result)`** — `json`, outbound. A `dict` in, a **string** out. You call this yourself, to make the tool message's `content` legal (Section 4.1). **The key point:** A message's `content` must be text, so a dict cannot go in it unconverted.
- **The SDK's own serialization of the whole body** — You never call it. It runs once on the finished request, which is what escapes the quotes of anything you already turned into a string.
- **`type_to_response_format_param(KBResponse)`** — `openai.lib`, outbound, internal. Your **class** in, a JSON Schema `dict` out. Run for you by `parse()`. Traced in `Exhaustive_3-tools.ipynb`, Section 6. **The key point:** This one carries no data at all — it is a description of a shape.

##### No wall — object and dict, both alive in your process

These cross nothing. They exist so you can read, log or index data you already have.

- **`completion.model_dump()`** — Pydantic. An **instance** in, a `dict` out. Types become `None`, quotes become single.
- **`completion.model_dump_json()`** — Pydantic. An **instance** in, a **string** out. The same data with `null` and double quotes.
- **`parsed.model_dump()`** — The same call on your own `KBResponse`.
- **`parsed.source`, `record["id"]`** — Plain attribute and key access. **The key point:** The dot works because it is an object; the brackets work because it is a dict. Reaching for the wrong one is the single most common symptom of having lost track of which form you are holding.

##### The one-sentence version

Ask "what have I got, and what does the next thing need?" — a file, a string, a dict or an object — and there is exactly one call in the list above that goes from the first to the second.

#### 8. Vocabulary, so the words stop colliding

Several words in this notebook mean two or three different things depending on which layer you are standing on, and the surrounding text rarely says which. Each one is defined here exactly once.

##### `object` — four meanings, and you meet all four in this script

This is the worst of them, so it goes first.

1. **A Python object.** Any value at all: a `dict` is an object, a `str` is an object, `completion` is an object. Used this way in "no object crosses the network".
2. **A JSON object.** What JSON calls a `{...}` block — the thing Python calls a `dict`. So "the request body is a JSON object" means "it is a set of key/value pairs in braces".
3. **`"type": "object"` in a JSON Schema.** Meaning 2 used as a type name: "this field holds a set of named sub-fields". You see it in `tools[0]["function"]["parameters"]["type"]` and at the top of `KBResponse`'s schema.
4. **`"object": "chat.completion"` in the response.** Neither of the above. Here `object` is an ordinary key name OpenAI chose, and its value is a label saying *what kind of thing this response is*. Pure coincidence of naming.

**How to tell them apart:** 1 is Python's word for "a value". 2 and 3 are JSON's word for "a dict". 4 is just a key called `object`.

##### `key` and `value`

- **Key**, in a dict or in JSON: The name half of a pair. `"id"` in `"id": 1`.
- **Value**: The other half. `1`.
- **API key**: Unrelated. A secret string that authenticates you. Shares the word with nothing else here.
- **"Use `source` as a key to look it up"**: Meaning 1, used loosely — a value that happens to identify a record. `record["id"]` is a key in that sense and a *value* in the dict sense, which is exactly why Section 5 is worth reading slowly.

##### `field`, `property`, `attribute`

Three words for "one named piece of a structure", each belonging to a different layer.

- **Field** — Pydantic's word. `answer` and `source` are the fields of `KBResponse`. Also the name of the thing `Field(description=...)` configures.
- **Property** — JSON Schema's word. The same two names appear under `"properties"` in the schema the SDK sends. (Python also has an unrelated `@property` decorator, which does not appear in this course.)
- **Attribute** — Python's word for something reached with a dot: `parsed.answer`. On a Pydantic object, the attributes *are* the fields.

So `answer` is a field of the class, a property in the schema, an attribute on the instance and a key in the dump — four words, one thing, four layers.

##### `parameters` and `arguments`

- **Parameters** — The names a function declares. `def search_kb(question: str)` has one parameter, `question`. Also the JSON Schema block in the tool definition, `"parameters"`, which describes exactly those names to the model.
- **Arguments** — The values actually passed in a call. `search_kb(question="What is the return policy?")` passes one argument. `tool_call.function.arguments` holds the arguments the model chose, as text.

##### `parse`, `load`, `dump`, `serialize`

- **Serialize / dump** — Turn a live structure into text. `json.dumps`, `model_dump_json`. (Confusingly, Pydantic's `model_dump` gives a `dict`, not text; `model_dump_json` is the one that gives text.)
- **Deserialize / parse / load** — The other direction: text into a live structure. `json.loads`, `json.load`, `model_validate_json`.
- **`client.chat.completions.parse(...)`** — Not quite either. It is `create()` plus two deserializations done for you afterwards: `message.parsed` and `parsed_arguments`.

##### `record`, `entry`, `item`, `message`

- **Record** — One `{"id": ..., "question": ..., "answer": ...}` dict in `kb.json`. The word comes from the data, not from Python.
- **Entry** — Used in these notebooks for one element of `messages`, numbered from 1 in the prose and from 0 in the API's error messages.
- **Item** — One element of any list, when nothing more specific fits.
- **Message** — One `{"role": ..., "content": ...}` dict. Note that `messages` (your list) and `message` (`completion.choices[0].message`, the model's reply object) are one letter apart and are not the same kind of thing at all — which is why `Exhaustive_3-tools.ipynb` names it `reply` instead.

##### `schema` and `data`

- **Schema** — A description of a shape, containing no values. `KBResponse`'s schema says `source` is an integer; it does not say `1`.
- **Data** — Actual values. `{"source": 1}`.
- Both travel as JSON text, which is why they are so easy to mix up on screen. `Exhaustive_2-structured.ipynb`, Section 7, is the long version of this distinction.

##### `content`

- **`message["content"]`** — The text of one chat message. In the tool message it happens to contain JSON, but the API treats it as plain text and nothing parses it for you.
- **`response.content`** — On an HTTP response object, the raw bytes of the reply. Different layer, same word. It appears in these notebooks only inside the fake-transport capture trick.

##### The one-sentence version

When a word feels overloaded, ask which layer you are on — your Python, the JSON text, the JSON Schema, or the API's own naming — and the meaning is decided.